In [1]:
#loading keys

from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
import os 
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [4]:
from openai import OpenAI

client = OpenAI()

In [6]:
message = [
    {
        "role" : "user",
        "content" : "Tell me a fun fact"
    }
]

response = client.chat.completions.create(model = "gpt-5.4-nano",messages =message)

reply = response.choices[0].message.content
print(reply)

Octopuses have three hearts—two pump blood to their gills, and the third pumps blood to the rest of their body. Also, when an octopus swims, the one pumping heart slows down, so they often prefer crawling over swimming.


In [7]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-nano",
    input="Write a one-sentence bedtime story about a unicorn.",
)

print(response.output_text)

In a moonlit meadow, a gentle unicorn tiptoed through the dew, sprinkling stardust on every sleeping flower until the whole night yawned softly into dreams.


## chain two calls

In [10]:
question_prompt = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."

question_response = client.responses.create(

    model = "gpt-5.4-mini",
    input = question_prompt
)

question = question_response.output_text
print(question)

A clock shows 3:15. What is the exact angle between the hour hand and the minute hand, and why does your method still work if the clock’s hour hand moves continuously rather than in discrete steps?


In [11]:
answer_response = client.responses.create(
    model="gpt-5.4-mini",
    input=question,
)

answer = answer_response.output_text
print(answer)

At **3:15**, the angle between the hour and minute hands is:

### 1) Minute hand
At 15 minutes, the minute hand points at the **3**, which is:
- \(15 \times 6^\circ = 90^\circ\) from 12.

### 2) Hour hand
The hour hand is at 3, but it has also moved a bit past 3 because 15 minutes have passed.
- Each hour mark is \(30^\circ\)
- In 15 minutes, the hour hand moves \(\frac{15}{60} \times 30^\circ = 7.5^\circ\)

So the hour hand is at:
- \(3 \times 30^\circ + 7.5^\circ = 97.5^\circ\)

### 3) Angle between them
\[
97.5^\circ - 90^\circ = 7.5^\circ
\]

## Answer:
**The exact angle is \(7.5^\circ\).**

## Why this still works with a continuously moving hour hand
This method works because it already accounts for continuous motion.

- The minute hand moves continuously, so at 15 minutes it is exactly at \(90^\circ\).
- The hour hand does **not** jump from hour to hour; it moves smoothly at a constant rate.
- So when we calculate the hour hand’s position as
  \[
  30^\circ \times \text{hour} + 0

In [12]:
from IPython.display import Markdown,display

display(Markdown(answer))

At **3:15**, the angle between the hour and minute hands is:

### 1) Minute hand
At 15 minutes, the minute hand points at the **3**, which is:
- \(15 \times 6^\circ = 90^\circ\) from 12.

### 2) Hour hand
The hour hand is at 3, but it has also moved a bit past 3 because 15 minutes have passed.
- Each hour mark is \(30^\circ\)
- In 15 minutes, the hour hand moves \(\frac{15}{60} \times 30^\circ = 7.5^\circ\)

So the hour hand is at:
- \(3 \times 30^\circ + 7.5^\circ = 97.5^\circ\)

### 3) Angle between them
\[
97.5^\circ - 90^\circ = 7.5^\circ
\]

## Answer:
**The exact angle is \(7.5^\circ\).**

## Why this still works with a continuously moving hour hand
This method works because it already accounts for continuous motion.

- The minute hand moves continuously, so at 15 minutes it is exactly at \(90^\circ\).
- The hour hand does **not** jump from hour to hour; it moves smoothly at a constant rate.
- So when we calculate the hour hand’s position as
  \[
  30^\circ \times \text{hour} + 0.5^\circ \times \text{minutes},
  \]
  we are measuring its exact continuous position at that moment.

So the formula gives the true angle whether the hour hand is treated as moving smoothly or just approximated in steps.

In [13]:
judge_prompt = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""

print(judge_prompt)


Here is a question:
A clock shows 3:15. What is the exact angle between the hour hand and the minute hand, and why does your method still work if the clock’s hour hand moves continuously rather than in discrete steps?

And here is a possible answer that might be correct or incorrect:
At **3:15**, the angle between the hour and minute hands is:

### 1) Minute hand
At 15 minutes, the minute hand points at the **3**, which is:
- \(15 \times 6^\circ = 90^\circ\) from 12.

### 2) Hour hand
The hour hand is at 3, but it has also moved a bit past 3 because 15 minutes have passed.
- Each hour mark is \(30^\circ\)
- In 15 minutes, the hour hand moves \(\frac{15}{60} \times 30^\circ = 7.5^\circ\)

So the hour hand is at:
- \(3 \times 30^\circ + 7.5^\circ = 97.5^\circ\)

### 3) Angle between them
\[
97.5^\circ - 90^\circ = 7.5^\circ
\]

## Answer:
**The exact angle is \(7.5^\circ\).**

## Why this still works with a continuously moving hour hand
This method works because it already accounts for 

In [14]:
judge_response = client.responses.create(
    model="gpt-5.4",
    input=judge_prompt,
)

print(judge_response.output_text)

The answer is **correct**.

Why:
- Minute hand at 15 minutes: \(15 \times 6^\circ = 90^\circ\)
- Hour hand at 3:15:  
  - At 3:00 it is at \(3 \times 30^\circ = 90^\circ\)
  - It moves \(30^\circ\) per hour, so in 15 minutes it moves \(\frac{15}{60}\times 30^\circ = 7.5^\circ\)
  - So its position is \(90^\circ + 7.5^\circ = 97.5^\circ\)

Thus the angle between the hands is:
\[
|97.5^\circ - 90^\circ| = 7.5^\circ
\]

The explanation about continuous motion is also correct:
- The hour hand moves continuously, not in jumps.
- The formula \(30h + 0.5m\) degrees exactly models that continuous motion.
- So the method remains valid for a real clock.

**Verdict: Correct.**


## three chained calls

In [15]:
# Call 1: pick a business area
prompt_1 = "Pick one business area that might be worth exploring for an agentic AI opportunity. Respond only with the name of the business area."

response_1 = client.responses.create(model="gpt-5.4-mini", input=prompt_1)
business_area = response_1.output_text
print(business_area)

Customer support


In [16]:
# Call 2: find a pain point in that area
prompt_2 = f"""
Business area: {business_area}

Describe one challenging pain point in this industry that might be ripe for an agentic AI solution. Respond with only the pain point, in two or three sentences.
"""

response_2 = client.responses.create(model="gpt-5.4-mini", input=prompt_2)
pain_point = response_2.output_text
print(pain_point)

Support teams struggle to resolve high volumes of repetitive, multi-step customer requests across email, chat, and internal systems while keeping response times low and answers consistent. A major pain point is the manual triage and context gathering required to route each issue correctly and pull together the right account, order, and policy information before an agent can take action.


In [17]:
# Call 3: propose the agentic solution
prompt_3 = f"""
Business area: {business_area}
Pain point: {pain_point}

Propose an agentic AI solution for this pain point. Say what the agent would do, which tools it would use, and how a human would stay in control.
"""

response_3 = client.responses.create(model="gpt-5.4-mini", input=prompt_3)
solution = response_3.output_text
display(Markdown(solution))

Here’s an agentic AI solution designed for customer support teams handling repetitive, multi-step requests across channels.

## Proposed solution: Support Triage and Resolution Agent

An AI agent sits in front of email, chat, and internal support tools to **triage incoming cases, gather context, draft or execute routine actions, and route exceptions to the right human agent**. Its goal is to reduce manual back-and-forth, speed up resolution, and keep responses consistent.

---

## What the agent would do

### 1. Ingest and classify incoming requests
The agent monitors inbound messages from:
- Email
- Chat
- Support ticketing systems

It identifies:
- Request type: refund, order status, address change, password reset, cancellation, warranty claim, etc.
- Urgency
- Sentiment
- Required handling path
- Whether the request is routine, policy-bound, or needs escalation

### 2. Gather relevant context automatically
Before a human agent touches the case, the AI pulls together the needed information:
- Customer profile and account history
- Open and past tickets
- Order details
- Shipping status
- Billing status
- Subscription or membership status
- Relevant policy rules or eligibility criteria

It can also ask clarifying questions if the request is incomplete.

### 3. Perform routine actions where allowed
For common, low-risk requests, the agent can complete actions directly, such as:
- Looking up order status
- Issuing a standard case update
- Creating a return request
- Resetting a password
- Updating a shipping address if policy allows
- Canceling a subscription within allowed rules
- Initiating a refund workflow for eligible cases

### 4. Draft responses for human review
For cases that require human approval, the agent prepares:
- A recommended next step
- A concise case summary
- A drafted response in the right tone and policy language
- Supporting evidence and system history

### 5. Route to the right team
If the issue is not resolvable automatically, the agent routes it based on:
- Issue type
- Customer tier
- Language
- Risk level
- Product line
- Policy exceptions needed

---

## Tools the agent would use

### Customer communication tools
- **Email inbox integration**: read and categorize support emails
- **Chat platform integration**: respond in real time or hand off to an agent
- **Ticketing system**: create, update, tag, and route cases in systems like Zendesk, Salesforce Service Cloud, Freshdesk, or ServiceNow

### Internal data tools
- **CRM access**: pull customer account and contact history
- **Order management system**: retrieve order, shipment, and fulfillment data
- **Billing/subscription system**: check invoices, renewals, refunds, and payment status
- **Knowledge base / policy repository**: retrieve approved procedures, eligibility rules, and macros

### Action tools
- **Workflow automation tools**: trigger return authorizations, refund requests, cancellations, escalation workflows
- **Identity verification tools**: confirm customer identity before sensitive actions
- **Case management APIs**: update ticket fields, assign queues, add internal notes

### AI support tools
- **Classification model**: detect issue type and urgency
- **Retrieval engine**: fetch relevant policy and account data
- **Reasoning/planning layer**: determine the right sequence of steps
- **Response generation layer**: draft consistent, policy-compliant replies

---

## How the agent would work in practice

### Example: “Where is my order?”
1. Customer sends email or chat.
2. Agent identifies intent as order status.
3. It looks up the customer account and order tracking data.
4. It checks whether the package is delayed or delivered.
5. If delivery is on track, it drafts a response with tracking info.
6. If delayed, it suggests compensation or escalation based on policy.
7. Human reviews if needed before sending.

### Example: “I need to change my shipping address”
1. Agent verifies the customer and checks order status.
2. It determines whether the order is still editable.
3. If allowed, it updates the address directly.
4. If not allowed, it routes the case to fulfillment or support with a prepared summary.

---

## How a human stays in control

The human-in-the-loop design is essential. The AI should assist, not fully replace, support agents for sensitive or ambiguous cases.

### Control mechanisms

#### 1. Approval thresholds
The agent can act automatically only for low-risk, predefined actions.
Examples:
- Safe to auto-resolve: order status, FAQ responses, simple account updates
- Requires approval: refunds above a threshold, cancellations after shipment, policy exceptions

#### 2. Confidence-based escalation
If the agent is uncertain about the intent, data quality, or policy fit, it must:
- Stop
- Summarize what it found
- Ask a human to confirm next action

#### 3. Editable drafts
Humans review and edit:
- Customer-facing responses
- Internal notes
- Suggested actions
- Escalation routing

#### 4. Audit trail
Every AI action is logged:
- What data it accessed
- What recommendation it made
- What actions were taken
- Who approved them

#### 5. Policy guardrails
The agent can only operate within rules set by the business:
- Refund limits
- Identity verification requirements
- Sensitive customer data handling
- Restricted actions for regulated cases

#### 6. Exception handling
Any unusual, emotional, high-value, or high-risk case goes to a human by default.

---

## Benefits

- Faster first response and resolution times
- Lower agent workload from repetitive triage
- More consistent answers across channels
- Better routing and fewer misassigned cases
- Improved agent productivity through pre-collected context
- Reduced customer frustration from repeated information requests

---

## Best-fit use cases
This works especially well for:
- Order and shipping inquiries
- Returns and refunds
- Account access issues
- Subscription changes
- Policy-based eligibility checks
- Multi-step support requests that require data from multiple systems

---

If you want, I can also turn this into:
1. a **one-page architecture diagram description**,  
2. a **workflow for a specific support scenario**, or  
3. a **business case with ROI metrics**.